# RNA-Seq End-to-End Workflow (Clean Notebook)

This notebook consolidates the full analysis pipeline:

- Load raw count data and sample annotations
- Run DESeq2 for normalization and differential expression
- Identify significant genes and attach gene symbols
- Build PCA embeddings (all genes, significant genes, pathway-weighted)
- Compare classification performance across embeddings
- Persist key outputs for downstream reuse

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_curve, roc_auc_score)

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import warnings
warnings.filterwarnings('ignore')

sns.set_context('notebook')
plt.style.use('seaborn-whitegrid')

SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results')
PATHWAY_DIR = RESULTS_DIR / 'pathway'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PATHWAY_DIR.mkdir(parents=True, exist_ok=True)

OVERWRITE_RESULTS = False

OSError: 'seaborn-whitegrid' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)

In [ ]:
def normalize_gene_ids(values):
    'Return a tidy Index of gene IDs without trailing decimals.'
    return (
        pd.Index(values)
        .astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )


def fetch_normalized_counts(dds, counts_df, metadata_df):
    'Extract normalized counts from a fitted DeseqDataSet.'
    norm_mat = None
    for attr in ('normed_counts', 'normalized_counts', 'norm_counts'):
        attr_val = getattr(dds, attr, None)
        if attr_val is not None:
            norm_mat = pd.DataFrame(attr_val)
            break
    if norm_mat is None:
        if hasattr(dds, 'size_factors'):
            sf = pd.Series(dds.size_factors, index=metadata_df.index)
        elif hasattr(dds, 'obs') and 'size_factors' in getattr(dds, 'obs').columns:
            sf = pd.Series(dds.obs['size_factors'], index=metadata_df.index)
        else:
            raise AttributeError('Could not locate normalized counts or size factors on the DESeq2 dataset.')
        norm_counts = counts_df.div(sf, axis=1)
    else:
        if norm_mat.shape[0] == metadata_df.shape[0]:
            norm_mat.index = metadata_df.index
            norm_counts = norm_mat.T
        else:
            norm_counts = norm_mat
    norm_counts.index = normalize_gene_ids(norm_counts.index)
    norm_counts.columns = metadata_df.index
    norm_counts = norm_counts.reindex(counts_df.index)
    return norm_counts


def compute_standard_pca(features, n_components, label_prefix='PC', random_state=SEED):
    'Run PCA on (samples x features) DataFrame and return scores DataFrame.'
    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)
    n_comp = max(1, min(n_components, scaled.shape[0], scaled.shape[1]))
    pca = PCA(n_components=n_comp, random_state=random_state)
    scores = pca.fit_transform(scaled)
    cols = [f"{label_prefix}{i+1}" for i in range(n_comp)]
    scores_df = pd.DataFrame(scores, index=features.index, columns=cols)
    return scores_df, pca


def compute_weighted_pca(features, weights, n_components, label_prefix='wPC', random_state=SEED):
    'Apply sqrt-weighted PCA on (samples x genes) features using per-gene weights.'
    shared = features.columns.intersection(weights.index)
    if shared.empty:
        raise ValueError('No overlap between expression columns and weight index.')
    weights_use = weights.loc[shared]
    expr_use = features.loc[:, shared]
    scaled = StandardScaler().fit_transform(expr_use)
    weighted = scaled * np.sqrt(weights_use.values)
    n_comp = max(1, min(n_components, weighted.shape[0], weighted.shape[1]))
    pca = PCA(n_components=n_comp, random_state=random_state)
    scores = pca.fit_transform(weighted)
    cols = [f"{label_prefix}{i+1}" for i in range(n_comp)]
    scores_df = pd.DataFrame(scores, index=expr_use.index, columns=cols)
    return scores_df, pca, weights_use


def run_random_forest(features, labels, approach_name, test_size=0.2, random_state=SEED):
    'Train/test split plus cross-validated AUC for a RandomForest classifier.'
    shared_samples = features.index.intersection(labels.index)
    X = features.loc[shared_samples].copy()
    y = labels.loc[shared_samples].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    clf = RandomForestClassifier(n_estimators=400, random_state=random_state, n_jobs=-1)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    metrics = {
        'approach': approach_name,
        'n_samples': X.shape[0],
        'n_features': X.shape[1],
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred, zero_division=0),
        'test_recall': recall_score(y_test, y_pred, zero_division=0),
        'test_f1': f1_score(y_test, y_pred, zero_division=0),
        'test_auc': roc_auc_score(y_test, y_proba),
        'roc_curve': (fpr, tpr),
        'y_test': y_test,
        'y_pred_proba': y_proba,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    cv_scores = cross_val_score(
        RandomForestClassifier(n_estimators=400, random_state=random_state, n_jobs=-1),
        X,
        y,
        cv=cv,
        scoring='roc_auc',
    )
    metrics['cv_auc_mean'] = cv_scores.mean()
    metrics['cv_auc_std'] = cv_scores.std()

    return metrics, clf


## Load and prepare raw inputs

In [ ]:
counts_path = DATA_DIR / 'GSE95640_raw_counts_GRCh38.p13_NCBI.tsv'
metadata_source = DATA_DIR / 'GSE95640_series_matrix.txt'
gene_annot_path = DATA_DIR / 'Human.GRCh38.p13.annot.tsv'

counts = pd.read_csv(counts_path, sep='	', index_col='GeneID', dtype=str)
counts.index = normalize_gene_ids(counts.index)
counts = counts.apply(pd.to_numeric, errors='coerce')
counts.columns = counts.columns.astype(str)

print(f'Counts matrix: {counts.shape[0]:,} genes × {counts.shape[1]} samples')

sample_matrix = pd.read_csv(
    metadata_source,
    sep='	',
    skiprows=48,
    nrows=44,
    index_col=0,
    header=0
)
sample_matrix = sample_matrix.iloc[[0, 11], :].T
sample_matrix.columns = ['sample_name', 'group_raw']
sample_matrix['sample_name'] = sample_matrix['sample_name'].astype(str)
sample_matrix['group'] = sample_matrix['group_raw'].str.strip()
sample_matrix['group_0_1'] = sample_matrix['group'].str.lower().str.contains('after 8 weeks').astype(int)
sample_matrix = sample_matrix.set_index('sample_name')

missing_samples = sorted(set(counts.columns) - set(sample_matrix.index))
if missing_samples:
    raise ValueError(f'Metadata missing for {len(missing_samples)} samples (first few: {missing_samples[:5]})')

metadata = sample_matrix.loc[counts.columns].copy()
metadata['condition'] = metadata['group_0_1'].map({0: 'baseline', 1: 'lcd'}).astype('category')

print(metadata['condition'].value_counts())

gene_annotation = pd.read_csv(gene_annot_path, sep='	', usecols=['GeneID', 'Symbol'], dtype=str)
gene_annotation['GeneID'] = normalize_gene_ids(gene_annotation['GeneID'])
gene_annotation = gene_annotation.drop_duplicates(subset='GeneID').set_index('GeneID')

metadata.head()

## Differential expression analysis with DESeq2

In [ ]:
dds = DeseqDataSet(counts=counts.T, metadata=metadata[['condition']], design_factors='condition')
dds.deseq2()

stats = DeseqStats(dds, contrast=['condition', 'lcd', 'baseline'])
stats.summary()

de_results = stats.results_df.copy()
de_results.index = normalize_gene_ids(de_results.index)

de_results = de_results.merge(gene_annotation, left_index=True, right_index=True, how='left')
de_results = de_results.rename(columns={'Symbol': 'gene_symbol'})
de_results = de_results.sort_values('padj')

norm_counts = fetch_normalized_counts(dds, counts, metadata[['condition']]).fillna(0)

norm_counts_path = RESULTS_DIR / 'GSE95640_DeSEQ2_normalized_counts.csv'
metadata_out_path = RESULTS_DIR / 'GSE95640_sample_annotation.csv'
de_results_path = RESULTS_DIR / 'DESeq2_all_results.csv'

metadata_out = metadata.copy()
metadata_out.insert(0, 'sample_name', metadata_out.index)

if OVERWRITE_RESULTS or not norm_counts_path.exists():
    norm_counts.to_csv(norm_counts_path)
if OVERWRITE_RESULTS or not metadata_out_path.exists():
    metadata_out.to_csv(metadata_out_path)
if OVERWRITE_RESULTS or not de_results_path.exists():
    de_results.to_csv(de_results_path)

print(f'Normalized counts shape: {norm_counts.shape}')
print(f'DE results rows: {de_results.shape[0]:,}')

de_results.head()

## Significant genes

In [ ]:
ALPHA = 0.05
LFC_THRESHOLD = 1.0

significant = de_results[(de_results['padj'] < ALPHA) & (de_results['log2FoldChange'].abs() > LFC_THRESHOLD)].copy()
significant = significant.sort_values('padj')

sig_path = RESULTS_DIR / 'DESeq2_significant_genes.csv'
if OVERWRITE_RESULTS or not sig_path.exists():
    significant.to_csv(sig_path)

print(f'Significant genes: {significant.shape[0]}')
significant.head()

## Prepare matrices for embeddings and classification

In [ ]:
labels = metadata['group_0_1'].astype(int)
labels.name = 'lcd_status'

expr_all = norm_counts.T
expr_all = expr_all.loc[:, expr_all.var(axis=0) > 0]

sig_gene_ids = significant.index.intersection(norm_counts.index)
expr_sig = norm_counts.loc[sig_gene_ids].T

sig_symbols = significant.loc[sig_gene_ids, 'gene_symbol'].fillna(sig_gene_ids)
expr_sig.columns = sig_symbols.values
expr_sig = expr_sig.groupby(expr_sig.columns, axis=1).mean()
expr_sig = expr_sig.loc[:, expr_sig.var(axis=0) > 0]

print(f'Samples: {expr_all.shape[0]}')
print(f'All genes (post variance filter): {expr_all.shape[1]}')
print(f'Significant genes (unique symbols): {expr_sig.shape[1]}')

expr_all.head()

### PCA baseline: all genes

In [ ]:
metrics_list = []

n_components_all = min(50, expr_all.shape[0] - 1, expr_all.shape[1])
all_scores, all_pca = compute_standard_pca(expr_all, n_components=n_components_all)
all_scores_path = RESULTS_DIR / 'all_genes_pca_scores_clean.csv'
if OVERWRITE_RESULTS or not all_scores_path.exists():
    all_scores.to_csv(all_scores_path)

metrics_all, _ = run_random_forest(all_scores, labels, f'All genes PCA ({all_scores.shape[1]} PCs)')
metrics_all['n_components'] = all_scores.shape[1]
metrics_all['notes'] = f"{expr_all.shape[1]} genes"
metrics_list.append(metrics_all)

print({k: round(metrics_all[k], 3) for k in ('test_accuracy', 'test_auc', 'cv_auc_mean', 'cv_auc_std')})

### PCA baseline: significant genes

In [ ]:
n_components_sig = min(15, expr_sig.shape[0] - 1, expr_sig.shape[1])
sig_scores, sig_pca = compute_standard_pca(expr_sig, n_components=n_components_sig)
sig_scores_path = RESULTS_DIR / 'significant_genes_pca_scores_clean.csv'
if OVERWRITE_RESULTS or not sig_scores_path.exists():
    sig_scores.to_csv(sig_scores_path)

metrics_sig, _ = run_random_forest(sig_scores, labels, f'Significant genes PCA ({sig_scores.shape[1]} PCs)')
metrics_sig['n_components'] = sig_scores.shape[1]
metrics_sig['notes'] = f"{expr_sig.shape[1]} gene symbols"
metrics_list.append(metrics_sig)

print({k: round(metrics_sig[k], 3) for k in ('test_accuracy', 'test_auc', 'cv_auc_mean', 'cv_auc_std')})

### Raw significant genes (no PCA)

In [ ]:
sig_standardized = pd.DataFrame(
    StandardScaler().fit_transform(expr_sig),
    index=expr_sig.index,
    columns=expr_sig.columns
)

metrics_sig_raw, _ = run_random_forest(sig_standardized, labels, 'Significant genes (raw features)')
metrics_sig_raw['n_components'] = sig_standardized.shape[1]
metrics_sig_raw['notes'] = 'raw standardized genes'
metrics_list.append(metrics_sig_raw)

print({k: round(metrics_sig_raw[k], 3) for k in ('test_accuracy', 'test_auc', 'cv_auc_mean', 'cv_auc_std')})

### Pathway-weighted PCA

In [ ]:
weights_file = PATHWAY_DIR / 'gene_weights.csv'
if weights_file.exists():
    gene_weights = pd.read_csv(weights_file, index_col=0)
    weight_series = gene_weights['final_weight'].astype(float)

    weighted_scores, weighted_pca, weights_used = compute_weighted_pca(
        expr_sig,
        weight_series,
        n_components=min(15, expr_sig.shape[0] - 1, expr_sig.shape[1])
    )

    weighted_scores_path = PATHWAY_DIR / 'weighted_pca_scores_clean.csv'
    if OVERWRITE_RESULTS or not weighted_scores_path.exists():
        weighted_scores.to_csv(weighted_scores_path)

    metrics_weighted, _ = run_random_forest(weighted_scores, labels, f'Pathway-weighted PCA ({weighted_scores.shape[1]} PCs)')
    metrics_weighted['n_components'] = weighted_scores.shape[1]
    metrics_weighted['notes'] = f"{len(weights_used)} weighted genes"
    metrics_list.append(metrics_weighted)

    print({k: round(metrics_weighted[k], 3) for k in ('test_accuracy', 'test_auc', 'cv_auc_mean', 'cv_auc_std')})
else:
    print('Weighted PCA skipped: gene_weights.csv not found in results/pathway.')

## Compare embeddings

In [ ]:
metrics_df = pd.DataFrame(metrics_list)
metrics_display_cols = [
    'approach',
    'n_components',
    'n_features',
    'test_accuracy',
    'test_auc',
    'cv_auc_mean',
    'cv_auc_std',
    'notes',
]

display(metrics_df[metrics_display_cols])

metrics_summary_path = RESULTS_DIR / 'clean_workflow_classification_metrics.csv'
metrics_df.to_csv(metrics_summary_path, index=False)
print(f'Saved summary to {metrics_summary_path}')


In [ ]:
plt.figure(figsize=(6, 6))
for entry in metrics_list:
    if 'roc_curve' not in entry:
        continue
    fpr, tpr = entry['roc_curve']
    plt.plot(fpr, tpr, lw=2, label=f"{entry['approach']} (AUC={entry['test_auc']:.2f})")

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC comparison across embeddings')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Wrap-up

In [ ]:
print(f'Significant genes retained: {significant.shape[0]}')
for entry in metrics_list:
    print(
        f"{entry['approach']}: test AUC={entry['test_auc']:.3f}, CV AUC={entry['cv_auc_mean']:.3f} ± {entry['cv_auc_std']:.3f}"
    )
print(f'Metrics table: {metrics_summary_path}')